# Teacher A — Whisper → IndicTrans2

## Copy/run order

1. Block 1 — configuration
2. Block 2 — Phase 0A runtime probe
3. Block 3 — Phase 0B bundle and audio verification
4. Block 4 — stop gate
5. Block 5 — dependencies (after approval)
6. Block 6 — shared utilities
7. Block 7 — immutable model revisions
8. Block 8 — Whisper ASR
9. Block 9 — IndicTrans2 translation
10. Block 10 — tokenizer-risk report
11. Block 11 — metadata

This notebook is independently runnable in a clean Colab GPU session. It benchmarks English speech → English transcript → Hindi text. It never trains Nadantara, changes the source manifest, or overwrites human Hindi references.

Run cells top-to-bottom. The first cells are a hard Phase 0 gate: runtime and uploaded-bundle access only. Do not run the dependency/model cells until that gate passes and the experiment is approved.

Official APIs: [Whisper](https://huggingface.co/openai/whisper-large-v3-turbo), [IndicTrans2](https://huggingface.co/ai4bharat/indictrans2-en-indic-1B). IndicTrans2 requires accepted Hugging Face access conditions and `trust_remote_code=True`; its revision is resolved and recorded before loading.

In [ ]:
# BLOCK 1 — configuration and benchmark bundle paths.
from __future__ import annotations

import gc
import hashlib
import json
import os
from pathlib import Path
import platform
import shutil
import statistics
import tarfile
import time
import unicodedata
import zlib

import numpy as np
import torch

MODEL_IDS = {
    'speech_teacher': 'openai/whisper-large-v3-turbo',
    'translation_teacher': 'ai4bharat/indictrans2-en-indic-1B',
}
BUNDLE_ARCHIVE = Path('/content/nadantara-benchmark-smoke.tar.gz')
BUNDLE_ROOT = Path('/content/nadantara-benchmark-smoke')
OUTPUT_ROOT = Path('/content/teacher_a_outputs')
RETRY_FAILED = False
MAX_AUDIO_SECONDS = 30.0
MAX_TRANSLATION_TOKENS = 256
print('python=', platform.python_version())
print('bundle_archive=', BUNDLE_ARCHIVE)
print('bundle_root=', BUNDLE_ROOT)
print('output_root=', OUTPUT_ROOT)

In [ ]:
# BLOCK 2 — PHASE 0A runtime probe; this cell downloads no models.
import shutil as _shutil

print('python=', platform.python_version())
print('torch=', torch.__version__)
print('torch_cuda_version=', torch.version.cuda)
print('cuda_available=', torch.cuda.is_available())
print('content_exists=', Path('/content').is_dir())
print('free_disk_gib=', round(_shutil.disk_usage('/content').free / 1024**3, 3))
if not torch.cuda.is_available():
    raise RuntimeError('PHASE 0 STOP: no CUDA GPU is available')
device_index = torch.cuda.current_device()
props = torch.cuda.get_device_properties(device_index)
free_bytes, total_bytes = torch.cuda.mem_get_info(device_index)
print('gpu_index=', device_index)
print('gpu_name=', props.name)
print('gpu_vram_gib=', round(props.total_memory / 1024**3, 3))
print('gpu_free_memory_gib=', round(free_bytes / 1024**3, 3))
print('gpu_total_memory_gib=', round(total_bytes / 1024**3, 3))
print('PHASE_0_RUNTIME_PASS')

In [ ]:
# BLOCK 3 — PHASE 0B uploaded-file access, manifest identity, and audio-byte checks.
if not BUNDLE_ARCHIVE.is_file():
    raise FileNotFoundError(f'upload the benchmark archive first: {BUNDLE_ARCHIVE}')
def _safe_extract(archive: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:gz') as tar:
        root = destination.parent.resolve()
        for member in tar.getmembers():
            target = (destination.parent / member.name).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f'unsafe archive member: {member.name}')
        tar.extractall(destination.parent)
if not (BUNDLE_ROOT / 'benchmark_manifest.jsonl').is_file():
    _safe_extract(BUNDLE_ARCHIVE, BUNDLE_ROOT)
manifest_path = BUNDLE_ROOT / 'benchmark_manifest.jsonl'
metadata_path = BUNDLE_ROOT / 'bundle_metadata.json'
rows = [json.loads(line) for line in manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]
if not rows:
    raise RuntimeError('benchmark manifest is empty')
if len({row['sample_id'] for row in rows}) != len(rows):
    raise RuntimeError('duplicate sample IDs in benchmark manifest')
def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while chunk := stream.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()
for row in rows:
    audio_path = BUNDLE_ROOT / row['audio_path']
    if not audio_path.is_file():
        raise FileNotFoundError(audio_path)
    actual_hash = _sha256(audio_path)
    if actual_hash != row['audio_sha256']:
        raise RuntimeError(f'audio hash mismatch for {row["sample_id"]}')
print('samples=', len(rows))
print('sample_ids=', [row['sample_id'] for row in rows])
print('bundle_metadata=', json.loads(metadata_path.read_text(encoding='utf-8')))
print('PHASE_0_FILES_PASS')

## BLOCK 4 — Phase 0 stop gate

At this point confirm `PHASE_0_RUNTIME_PASS` and `PHASE_0_FILES_PASS`, and inspect the printed GPU, VRAM, sample IDs, and hashes. Stop here unless the teacher smoke run is explicitly approved. The next cell installs teacher-specific dependencies; it does not run a model by itself.

In [ ]:
# BLOCK 5 — POST-APPROVAL ONLY — Teacher A's isolated dependencies.
%pip install -q 'transformers==4.53.2' accelerate sentencepiece IndicTransToolkit soundfile scipy psutil huggingface_hub sacrebleu

In [ ]:
# BLOCK 6 — shared Teacher A utilities.
import gc
import math
import psutil
import soundfile as sf
from scipy.signal import resample_poly

TARGET_RATE = 16_000
def _token() -> str | None:
    value = os.environ.get('HF_TOKEN')
    try:
        from google.colab import userdata
        value = value or userdata.get('HF_TOKEN')
    except Exception:
        pass
    return value or None
HF_TOKEN = _token()
def normalize_hindi(text: str) -> str:
    return ' '.join(unicodedata.normalize('NFC', text).split())
def audio_sha256(path: Path) -> str:
    return _sha256(path)
def read_audio(path: Path) -> tuple[np.ndarray, float, int]:
    array, sample_rate = sf.read(path, dtype='float32', always_2d=True)
    waveform = array.mean(axis=1) if array.shape[1] > 1 else array[:, 0]
    if sample_rate != TARGET_RATE:
        divisor = math.gcd(int(sample_rate), TARGET_RATE)
        waveform = resample_poly(waveform, TARGET_RATE // divisor, int(sample_rate) // divisor).astype('float32')
        sample_rate = TARGET_RATE
    waveform = np.clip(waveform, -1.0, 1.0).astype('float32')
    duration = len(waveform) / TARGET_RATE
    if duration > MAX_AUDIO_SECONDS:
        raise ValueError(f'audio exceeds {MAX_AUDIO_SECONDS}s: {duration:.3f}s')
    return waveform, duration, int(sample_rate)
def existing_records(path: Path) -> dict[str, dict[str, object]]:
    if not path.is_file(): return {}
    records = {}
    for line in path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            row = json.loads(line); records[str(row['sample_id'])] = row
    return records
def append_jsonl(path: Path, row: dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n')
        stream.flush()
def clear_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
print('transformers=', __import__('transformers').__version__)
print('HF_TOKEN_available=', bool(HF_TOKEN))
durations = []
for row in rows:
    try:
        _, duration, _ = read_audio(BUNDLE_ROOT / row['audio_path'])
        durations.append(duration)
    except Exception as exc:
        print('duration_error=', row['sample_id'], repr(exc))
if durations:
    print('duration_min_median_p90_p95_max=', min(durations), statistics.median(durations), np.percentile(durations, 90), np.percentile(durations, 95), max(durations))

In [ ]:
# BLOCK 7 — POST-APPROVAL ONLY — resolve and record immutable model revisions.
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
MODEL_REVISIONS = {model_id: api.model_info(model_id, token=HF_TOKEN).sha for model_id in MODEL_IDS.values()}
print(json.dumps(MODEL_REVISIONS, indent=2))

In [ ]:
# BLOCK 8 — POST-APPROVAL ONLY — Whisper ASR stage. One sample per checkpointed JSONL record.
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

asr_path = OUTPUT_ROOT / 'whisper_asr.jsonl'
asr_records = existing_records(asr_path)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
whisper_processor = AutoProcessor.from_pretrained(MODEL_IDS['speech_teacher'], revision=MODEL_REVISIONS[MODEL_IDS['speech_teacher']], token=HF_TOKEN)
whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_IDS['speech_teacher'], revision=MODEL_REVISIONS[MODEL_IDS['speech_teacher']], torch_dtype=dtype, low_cpu_mem_usage=True, use_safetensors=True, token=HF_TOKEN).to(device)
whisper_model.eval()
whisper_generation_config = {'language': 'english', 'task': 'transcribe', 'do_sample': False, 'num_beams': 1, 'max_new_tokens': 256}
for row in rows:
    sample_id = row['sample_id']
    if sample_id in asr_records and (asr_records[sample_id].get('status') == 'ok' or not RETRY_FAILED): continue
    base = {'schema_version': 1, 'sample_id': sample_id, 'audio_path': row['audio_path'], 'audio_sha256': row['audio_sha256'], 'source_manifest_sha256': row.get('source_manifest_sha256'), 'source_dataset': row.get('source_dataset'), 'source_dataset_revision': row.get('source_dataset_revision'), 'source_license': row.get('source_license'), 'source_url': row.get('source_url'), 'source_split': row.get('source_split'), 'reference_hindi': row.get('reference_hindi'), 'source_text': row.get('source_text'), 'source_asr_text': row.get('source_asr_text'), 'speech_teacher': MODEL_IDS['speech_teacher'], 'speech_teacher_revision': MODEL_REVISIONS[MODEL_IDS['speech_teacher']], 'speech_teacher_license': 'MIT', 'status': 'error', 'stage': 'whisper'}
    started = time.perf_counter()
    try:
        waveform, duration, _ = read_audio(BUNDLE_ROOT / row['audio_path'])
        inputs = whisper_processor(waveform, sampling_rate=TARGET_RATE, return_tensors='pt', return_attention_mask=True)
        inputs = {key: (value.to(device=device, dtype=dtype) if torch.is_floating_point(value) else value.to(device)) for key, value in inputs.items()}
        forced_ids = whisper_processor.get_decoder_prompt_ids(language='english', task='transcribe')
        with torch.inference_mode():
            generated = whisper_model.generate(**inputs, forced_decoder_ids=forced_ids, do_sample=False, num_beams=1, max_new_tokens=256, return_dict_in_generate=True, output_scores=True)
        transcript = whisper_processor.batch_decode(generated.sequences, skip_special_tokens=True)[0].strip()
        mean_logprob = None
        if generated.scores:
            transition = whisper_model.compute_transition_scores(generated.sequences, generated.scores, normalize_logits=True)
            finite = transition[0][torch.isfinite(transition[0])]
            mean_logprob = float(finite.mean().cpu()) if finite.numel() else None
        base.update({'status': 'ok', 'stage': None, 'whisper_transcript_en': transcript, 'asr_seconds': time.perf_counter() - started, 'duration_seconds': duration, 'whisper_mean_logprob': mean_logprob, 'whisper_generation_config': whisper_generation_config})
    except Exception as exc:
        base.update({'error': repr(exc), 'asr_seconds': time.perf_counter() - started})
    append_jsonl(asr_path, base); asr_records[sample_id] = base
    print(sample_id, base['status'], base.get('error', ''))
del whisper_model, whisper_processor
clear_cuda()
print('asr_output=', asr_path)

In [ ]:
# BLOCK 9 — POST-APPROVAL ONLY — IndicTrans2 text translation stage.
from IndicTransToolkit.processor import IndicProcessor
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

teacher_path = OUTPUT_ROOT / 'teacher_whisper_indictrans2.jsonl'
teacher_records = existing_records(teacher_path)
successful_asr = [row for row in rows if asr_records.get(row['sample_id'], {}).get('status') == 'ok']
if not successful_asr:
    raise RuntimeError('No successful Whisper ASR records. Rerun the ASR cell with RETRY_FAILED=True before starting IndicTrans2.')
print('successful_asr=', len(successful_asr), 'requested=', len(rows))
translation_id = MODEL_IDS['translation_teacher']
translation_revision = MODEL_REVISIONS[translation_id]
translation_tokenizer = AutoTokenizer.from_pretrained(translation_id, revision=translation_revision, trust_remote_code=True, token=HF_TOKEN)
translation_device = 'cuda' if torch.cuda.is_available() else 'cpu'
translation_model = AutoModelForSeq2SeqLM.from_pretrained(translation_id, revision=translation_revision, trust_remote_code=True, attn_implementation='eager', low_cpu_mem_usage=True, token=HF_TOKEN).to(translation_device)
if translation_device == 'cuda': translation_model.half()
translation_model.eval()
indic_processor = IndicProcessor(inference=True)
translation_generation_config = {'src_lang': 'eng_Latn', 'tgt_lang': 'hin_Deva', 'do_sample': False, 'num_beams': 5, 'num_return_sequences': 1, 'max_length': MAX_TRANSLATION_TOKENS, 'batch_size': 1}
for row in rows:
    sample_id = row['sample_id']; asr = asr_records.get(sample_id, {})
    if sample_id in teacher_records and (teacher_records[sample_id].get('status') == 'ok' or not RETRY_FAILED): continue
    base = {'schema_version': 1, 'sample_id': sample_id, 'audio_path': row['audio_path'], 'audio_sha256': row['audio_sha256'], 'source_manifest_sha256': row.get('source_manifest_sha256'), 'source_dataset': row.get('source_dataset'), 'source_dataset_revision': row.get('source_dataset_revision'), 'source_license': row.get('source_license'), 'source_url': row.get('source_url'), 'source_split': row.get('source_split'), 'reference_hindi': row.get('reference_hindi'), 'source_text': row.get('source_text'), 'source_asr_text': row.get('source_asr_text'), 'whisper_transcript_en': asr.get('whisper_transcript_en'), 'speech_teacher': MODEL_IDS['speech_teacher'], 'speech_teacher_revision': MODEL_REVISIONS[MODEL_IDS['speech_teacher']], 'speech_teacher_license': 'MIT', 'translation_teacher': translation_id, 'translation_teacher_revision': translation_revision, 'translation_teacher_license': 'MIT', 'teacher_pipeline': 'whisper-large-v3-turbo+indictrans2-en-indic-1B', 'status': 'error', 'stage': 'indictrans2', 'generation_config': translation_generation_config}
    started = time.perf_counter()
    if asr.get('status') != 'ok':
        base.update({'stage': 'whisper', 'error': asr.get('error', 'missing ASR output'), 'asr_seconds': asr.get('asr_seconds')})
    else:
        try:
            batch = indic_processor.preprocess_batch([str(asr['whisper_transcript_en'])], src_lang='eng_Latn', tgt_lang='hin_Deva')
            inputs = translation_tokenizer(batch, truncation=True, padding='longest', return_tensors='pt', return_attention_mask=True).to(translation_device)
            with torch.inference_mode():
                generated = translation_model.generate(**inputs, use_cache=True, min_length=0, max_length=MAX_TRANSLATION_TOKENS, num_beams=5, num_return_sequences=1)
            decoded = translation_tokenizer.batch_decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=True)
            postprocessed = indic_processor.postprocess_batch(decoded, lang='hin_Deva')[0]
            base.update({'status': 'ok', 'stage': None, 'teacher_hindi_model_decoded_raw': decoded[0], 'teacher_hindi_raw': postprocessed, 'teacher_hindi_normalized': normalize_hindi(postprocessed), 'asr_seconds': asr.get('asr_seconds'), 'translation_seconds': time.perf_counter() - started})
        except Exception as exc:
            base.update({'error': repr(exc), 'translation_seconds': time.perf_counter() - started, 'asr_seconds': asr.get('asr_seconds')})
    append_jsonl(teacher_path, base); teacher_records[sample_id] = base
    print(sample_id, base['status'], base.get('stage', 'ok'), base.get('error', ''))
del translation_model, translation_tokenizer, indic_processor
clear_cuda()
print('canonical_output=', teacher_path)

In [ ]:
# BLOCK 10 — optional pre-distillation tokenizer-risk report. Upload the exact baseline hindi_2k.model first.
TOKENIZER_PATH = Path('/content/hindi_2k.model')
MAX_SEMANTIC_TOKENS = 128
if TOKENIZER_PATH.is_file():
    import sentencepiece as spm
    tokenizer = spm.SentencePieceProcessor(model_file=str(TOKENIZER_PATH))
    output_rows = [json.loads(line) for line in teacher_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    lengths = []; unknown = 0; total = 0
    for row in output_rows:
        if row.get('status') != 'ok': continue
        ids = tokenizer.encode(str(row.get('teacher_hindi_raw', '')), out_type=int)
        lengths.append(len(ids)); unknown += sum(token == tokenizer.unk_id() for token in ids); total += len(ids)
    print({'vocab_size': tokenizer.vocab_size(), 'unknown_rate': unknown / max(1, total), 'p50': np.percentile(lengths, 50) if lengths else None, 'p90': np.percentile(lengths, 90) if lengths else None, 'p95': np.percentile(lengths, 95) if lengths else None, 'p99': np.percentile(lengths, 99) if lengths else None, 'max': max(lengths) if lengths else None, 'exceeding_max_semantic_tokens': sum(length + 1 > MAX_SEMANTIC_TOKENS for length in lengths) / max(1, len(lengths))})
else:
    print('tokenizer report skipped: upload the exact baseline tokenizer to', TOKENIZER_PATH)

In [ ]:
# BLOCK 11 — persist reproducibility metadata without secrets.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
metadata = {
    'schema_version': 1,
    'teacher_pipeline': 'whisper-large-v3-turbo+indictrans2-en-indic-1B',
    'model_ids': MODEL_IDS,
    'model_revisions': globals().get('MODEL_REVISIONS', {}),
    'model_licenses': {'openai/whisper-large-v3-turbo': 'MIT', 'ai4bharat/indictrans2-en-indic-1B': 'MIT'},
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'transformers': __import__('transformers').__version__,
    'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'precision': str(dtype),
    'bundle_metadata': json.loads((BUNDLE_ROOT / 'bundle_metadata.json').read_text(encoding='utf-8')),
    'generation_config': {'whisper': globals().get('whisper_generation_config', {}), 'indictrans2': globals().get('translation_generation_config', {})},
    'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
(OUTPUT_ROOT / 'experiment_metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(metadata, indent=2, ensure_ascii=False, sort_keys=True))
print('Teacher A complete; preserve teacher_whisper_indictrans2.jsonl and do not mix it with Teacher B until comparison.')